# BarGPT v1 checkpoint validation (small, bounded)

Load a checkpoint (preferred from shared-workstation runtime), rebuild a small ClickHouse validation panel, then evaluate and print checkpoint load context and validation metrics.

In [8]:
from __future__ import annotations

import sys
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
import time

import torch

# Local repo path and shared-workstation runtime root
REPO_ROOT = Path(r'D:/TradingCodes/quant-research-workbench')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.bar_gpt.v1.inference import load_pretrained
from research.bar_gpt.v1.config import DataConfig, ExperimentConfig, TrainConfig
from research.bar_gpt.v1.loader import ClickHouseBarStreamConfig, BarGPTIterableDataset, make_dataloader
from research.bar_gpt.v1.train import validate, _stream_config

from research.bar_gpt.v1.data import TARGET_NAMES
from research.mlops.env import load_env_files
from research.mlops.clickhouse import discover_clickhouse_env_files

load_env_files(discover_clickhouse_env_files(), verbose=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

# Shared-drive discovery and manual override
SHARED_TRAIN_ROOT = Path(r'\\DESKTOP-SAAI85T\Workstation-D\TradingML\runtimes\bar_gpt\v1\train')
LOCAL_TRAIN_ROOT = Path(r'D:\TradingML\runtimes\bar_gpt\v1\train')
CHECKPOINT_CANDIDATE_PATTERN = 'checkpoint_latest.pt'
CHECKPOINT_OVERRIDE = Path('')

def find_latest_checkpoint() -> Optional[Path]:
    for base in (CHECKPOINT_OVERRIDE, CHECKPOINT_OVERRIDE.parent if CHECKPOINT_OVERRIDE.is_file() else None):
        if base and base.is_file() and base.name == 'checkpoint_latest.pt':
            return base

    candidates = []
    for base in (SHARED_TRAIN_ROOT, LOCAL_TRAIN_ROOT):
        if not base.exists():
            continue
        candidates.extend(base.rglob(CHECKPOINT_CANDIDATE_PATTERN))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates:
        return None
    candidates.sort(key=lambda p: (p.stat().st_mtime, str(p)))
    return candidates[-1]

checkpoint_path = find_latest_checkpoint()
if checkpoint_path is None:
    raise FileNotFoundError('No checkpoint_latest.pt found in shared/laptop runtime roots; set CHECKPOINT_OVERRIDE.')
print('checkpoint:', checkpoint_path)
loaded_at_utc = datetime.now(timezone.utc).isoformat()
stat = checkpoint_path.stat()
checkpoint_asof_utc = datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat()
print('checkpoint as_of_utc:', checkpoint_asof_utc)
print('checkpoint bytes:', stat.st_size)
print('loaded_at_utc:', loaded_at_utc)

Loaded .env files: D:\TradingCodes\quant-research-workbench\.env
device: cuda
checkpoint: \\DESKTOP-SAAI85T\Workstation-D\TradingML\runtimes\bar_gpt\v1\train\bar-gpt-v1-20260803-214145\checkpoints\checkpoint_latest.pt
checkpoint as_of_utc: 2026-08-04T16:50:06.507234+00:00
checkpoint bytes: 163984679
loaded_at_utc: 2026-08-04T16:52:24.049784+00:00


In [9]:
# Load model, data contract, and checkpoint runtime counters
model, data_config, ckpt_payload = load_pretrained(checkpoint_path, device=DEVICE)
model_config = model.config
samples_seen = int(ckpt_payload.get('samples_seen', 0))
optimizer_steps = int(ckpt_payload.get('optimizer_steps', 0))
batches_seen = int(ckpt_payload.get('batches_seen', 0))
blocks_seen = int(ckpt_payload.get('blocks_seen', 0))
validation_runs_in_epoch = int(ckpt_payload.get('validation_runs_in_epoch', -1))
last_validation_samples = int(ckpt_payload.get('last_validation_samples', -1))
print('checkpoint contract hash:', ckpt_payload.get('contract_hash', 'n/a'))
print('plan_hash:', ckpt_payload.get('plan_hash', 'n/a'))
print('checkpoint loaded counters: samples_seen=', samples_seen, 'optimizer_steps=', optimizer_steps,
      'batches_seen=', batches_seen, 'blocks_seen=', blocks_seen)
print('validation_runs_in_epoch=', validation_runs_in_epoch, 'last_validation_samples=', last_validation_samples)
print('model quantiles:', tuple(model_config.quantiles), 'target_dim=', model_config.target_dim, 'feature_dim=', model_config.feature_dim)
print('ticker universe:', len(data_config.tickers), 'validation ticks:', len(data_config.validation_slices))
target_names = tuple(TARGET_NAMES)
print('target_names count:', len(target_names), 'first:', target_names[:8])

checkpoint contract hash: af9c4a0a64d7db0cac77ce0b86d757725c2d4cde417cba1583d9e81d3761769b
plan_hash: f592ccdf9978448e366d3780fb2414aff4d0bdc0867e01d437606f7b7963f2df
checkpoint loaded counters: samples_seen= 369098752 optimizer_steps= 96119 batches_seen= 96119 blocks_seen= 96119
validation_runs_in_epoch= -1 last_validation_samples= -1
model quantiles: (0.1, 0.5, 0.9) target_dim= 14 feature_dim= 46
ticker universe: 98 validation ticks: 8
target_names count: 14 first: ('endpoint_return', 'upper_excursion', 'lower_excursion', 'realized_volatility', 'log_trade_volume', 'log_trade_count', 'trade_available', 'bid_available')


In [10]:
# Build a small, bounded validation panel over 2026 data from clickhouse
EVAL_TICKERS = ('AAPL', 'MSFT', 'META')
EVAL_START_DATE = '2026-01-02'
EVAL_END_DATE = '2026-01-10'
VALIDATION_BLOCKS_PER_SLICE = 2
EVAL_VALIDATION_BATCHES = 12
EVAL_BATCH_SIZE = 2
EVAL_LOADER_WORKERS = 0

eval_tickers = tuple(t for t in EVAL_TICKERS if t in data_config.tickers)
if not eval_tickers:
    raise ValueError(f'No overlap between EVAL_TICKERS and configured cohort. candidates={EVAL_TICKERS}')

eval_validation_slices = tuple((t, EVAL_START_DATE, EVAL_END_DATE) for t in eval_tickers)
eval_data_config = replace(
    data_config,
    tickers=eval_tickers,
    validation_slices=eval_validation_slices,
    validation_start_date=EVAL_START_DATE,
    validation_blocks_per_slice=VALIDATION_BLOCKS_PER_SLICE,
    batch_size=EVAL_BATCH_SIZE,
    loader_workers=EVAL_LOADER_WORKERS,
    worker_prefetch_batches=1,
    coverage_mode='sequential',
)

eval_data_config.validate()
model_config.validate()
print('eval slices:', eval_validation_slices)
print('loader workers:', eval_data_config.loader_workers, 'batch_size:', eval_data_config.batch_size,
      'validation_blocks_per_slice:', eval_data_config.validation_blocks_per_slice)

# Use a small validate-only training contract.
eval_train_config = TrainConfig(
    validation_batches=EVAL_VALIDATION_BATCHES,
    cuda_prefetch=False,
    amp=False,
)
eval_train_config.validate()

eval_config = ExperimentConfig(model=model_config, data=eval_data_config, train=eval_train_config)

stream = _stream_config(eval_data_config)
if not isinstance(stream, ClickHouseBarStreamConfig):
    raise RuntimeError('Stream config was not ClickHouseBarStreamConfig')

validation_dataset = BarGPTIterableDataset(data_config=eval_data_config, stream_config=stream, split='validation', seed=eval_train_config.seed)
validation_loader = make_dataloader(validation_dataset, eval_data_config, drop_last=False)
print('validation loader ready')


eval slices: (('AAPL', '2026-01-02', '2026-01-10'), ('MSFT', '2026-01-02', '2026-01-10'), ('META', '2026-01-02', '2026-01-10'))
loader workers: 0 batch_size: 2 validation_blocks_per_slice: 2
validation loader ready


In [11]:
# Evaluate and print validation results
start = time.perf_counter()
with torch.no_grad():
    validation_results = validate(model, validation_loader, eval_config, DEVICE)
elapsed_sec = time.perf_counter() - start

print('validation run complete')
print('evaluation wall-time (sec):', round(elapsed_sec, 2))
print('checkpoint context at load -> samples_seen:', samples_seen, 'optimizer_steps:', optimizer_steps)
print('checkpoint as_of_utc:', checkpoint_asof_utc)
print('loaded_at_utc:', loaded_at_utc)

for key in sorted(validation_results):
    print(f'{key}: {validation_results[key]:.6f}')

validation run complete
evaluation wall-time (sec): 11.49
checkpoint context at load -> samples_seen: 369098752 optimizer_steps: 96119
checkpoint as_of_utc: 2026-08-04T16:50:06.507234+00:00
loaded_at_utc: 2026-08-04T16:52:24.049784+00:00
val/batches: 3.000000
val/horizon_300s_binary_brier: 0.001357
val/horizon_300s_coverage_q0.1: 0.268032
val/horizon_300s_coverage_q0.5: 0.632731
val/horizon_300s_coverage_q0.9: 0.868510
val/horizon_300s_luld_limit_state_within_horizon_positives: 0.000000
val/horizon_300s_median_mae: 0.443317
val/horizon_300s_sign_accuracy: 0.549357
val/horizon_30s_binary_brier: 0.063585
val/horizon_30s_coverage_q0.1: 0.124945
val/horizon_30s_coverage_q0.5: 0.512993
val/horizon_30s_coverage_q0.9: 0.873423
val/horizon_30s_luld_limit_state_within_horizon_positives: 0.000000
val/horizon_30s_median_mae: 0.411391
val/horizon_30s_sign_accuracy: 0.464722
val/horizon_3600s_binary_brier: 0.000001
val/horizon_3600s_coverage_q0.1: 0.311863
val/horizon_3600s_coverage_q0.5: 0.669373
